# 03 - FedAvg di ruang SFM (Paper 4, Fase 2, skenario-A)

**Tujuan:** simulasikan **FedAvg** 2 klien {CIC, UNSW} di ruang **9 fitur SFM**,
lalu bandingkan dengan dua batas: **centralized** (upper bound) dan **local-only**
(lower bound). Ini isi **Tabel D** di `documentation_federated.md`.

**Konsistensi kritis:** notebook ini **mengimpor logika MLP/FedAvg dari
`../aws/fed_node.py`** (init_weights, forward, train_local, fedavg, z-score).
Jadi simulasi di sini **identik** dengan yang dijalankan node EC2 di Fase 2 -
satu-satunya beda: transport bobot in-memory (bukan via S3). Tak ada logika FL
yang ditulis-ulang -> hasil notebook valid sebagai prediksi hasil EC2.

**Privasi:** z-score **per-klien lokal** (opsi (a), Bagian A.1); klien tak berbagi
data mentah maupun statistik. Centralized hanya upper-bound, bukan skenario privasi.

**Output:** `fed_out/fedavg_sfm.csv` (MCC/recall/precision/F1/bal-acc untuk
centralized / local-only / federated-SFM, dievaluasi di CIC-test, UNSW-test, GLOBAL-test).

> Jalankan setelah nb01 (butuh `fed_out/*.npz`). JANGAN mengarang angka.

## 0. Setup + impor logika FL dari fed_node.py

In [ ]:
import importlib, sys, subprocess
for m in ('numpy','pandas','scikit-learn'):
    if importlib.util.find_spec(m.replace('scikit-learn','sklearn')) is None:
        subprocess.run([sys.executable,'-m','pip','install','-q',m],check=True)
import os, numpy as np, pandas as pd
from sklearn.metrics import (matthews_corrcoef, recall_score, precision_score,
                             f1_score, balanced_accuracy_score)

# Impor logika FL yang SAMA dgn node EC2 (sumber tunggal kebenaran).
AWS_DIR=os.path.abspath(os.path.join('..','aws'))
if AWS_DIR not in sys.path: sys.path.insert(0, AWS_DIR)
import importlib.util as _u
_spec=_u.spec_from_file_location('fed_node', os.path.join(AWS_DIR,'fed_node.py'))
fed=_u.module_from_spec(_spec); _spec.loader.exec_module(fed)

OUTDIR='fed_out'; os.makedirs(OUTDIR, exist_ok=True)
ROUNDS=50; LOCAL_EPOCHS=1; LR=1e-3; SEED=42
print('fed_node loaded | LAYERS =', fed.LAYERS)
print('reuse:', [f for f in ('init_weights','forward','train_local','fedavg','zscore_fit','zscore_apply') if hasattr(fed,f)])

## 1. Muat partisi SFM per-klien (dari nb01)

Tiap klien = partisi SENDIRI. z-score **lokal** (fit di train klien, pakai
fungsi `fed.zscore_fit/apply` yang identik dgn EC2).

In [ ]:
def load(name):
    p=os.path.join(OUTDIR,name)
    if not os.path.exists(p): print('MISSING',p); return None,None
    d=np.load(p, allow_pickle=True); return np.asarray(d['X'],float), np.asarray(d['y'],int)
Xu_tr,yu_tr=load('unsw_train.npz'); Xu_te,yu_te=load('unsw_test.npz')
Xc_tr,yc_tr=load('cic_train.npz');  Xc_te,yc_te=load('cic_test.npz')
ok=all(v is not None for v in [Xu_tr,Xc_tr,Xu_te,Xc_te])
if ok:
    print('CIC  train',Xc_tr.shape,'pos',round(float(yc_tr.mean()),3))
    print('UNSW train',Xu_tr.shape,'pos',round(float(yu_tr.mean()),3))
else:
    print('(partisi belum ada - jalankan nb01 dulu di SageMaker)')

## 2. Helper evaluasi + prediksi MLP (pakai forward dari fed_node)

Prediksi = `fed.forward(W, Xz)[0] >= 0.5`. Scaler z-score **milik klien yang
melatih** (untuk model federated/centralized dipakai scaler yang sesuai).

In [ ]:
def metrics(y,yp):
    return dict(MCC=round(matthews_corrcoef(y,yp),4),
                recall=round(recall_score(y,yp,zero_division=0),4),
                precision=round(precision_score(y,yp,zero_division=0),4),
                F1=round(f1_score(y,yp,zero_division=0),4),
                bal_acc=round(balanced_accuracy_score(y,yp),4))
def predict(W, X, mu, sd):
    Xz=fed.zscore_apply(X, mu, sd)
    out,_=fed.forward(W, Xz)
    return (out>=0.5).astype(int)
res=[]
# evalsets global didefinisikan SEKALI di sini (dipakai centralized/local-only/federated)
if ok:
    Xg_te=np.vstack([Xc_te,Xu_te]); yg_te=np.concatenate([yc_te,yu_te])
    evalsets={'CIC-test':(Xc_te,yc_te),'UNSW-test':(Xu_te,yu_te),'GLOBAL-test':(Xg_te,yg_te)}

## 3. Baseline A - CENTRALIZED (upper bound)

Gabung train CIC+UNSW, satu scaler global, latih MLP yang SAMA (via
`fed.train_local` banyak epoch pada seluruh data). HANYA upper-bound, bukan privasi.

In [ ]:
if ok:
    Xtr=np.vstack([Xc_tr,Xu_tr]); ytr=np.concatenate([yc_tr,yu_tr])
    mu_c,sd_c=fed.zscore_fit(Xtr); Xz=fed.zscore_apply(Xtr,mu_c,sd_c)
    Wc=fed.init_weights(SEED)
    # 'epoch terpusat' ~ ROUNDS*LOCAL_EPOCHS agar jumlah update sebanding dgn federated
    Wc=fed.train_local(Wc, Xz, ytr, epochs=ROUNDS*LOCAL_EPOCHS, lr=LR, mu=0.0, seed=SEED)
    for nm,(Xe,ye) in evalsets.items():
        res.append({'setting':'centralized','eval':nm, **metrics(ye, predict(Wc,Xe,mu_c,sd_c))})
    print('centralized selesai')
print('=== SEL 3 (centralized) SELESAI ===')

## 4. Baseline B - LOCAL-ONLY (lower bound)

Tiap klien latih MLP HANYA di datanya sendiri (z-score lokal), tanpa federasi.
Evaluasi tiap model lokal di ketiga test set -> lihat kegagalan lintas-domain.

In [ ]:
if ok:
    locals_models={}
    for cid,(Xl,yl) in {'cic':(Xc_tr,yc_tr),'unsw':(Xu_tr,yu_tr)}.items():
        mu_l,sd_l=fed.zscore_fit(Xl); Xz=fed.zscore_apply(Xl,mu_l,sd_l)
        Wl=fed.init_weights(SEED)
        Wl=fed.train_local(Wl, Xz, yl, epochs=ROUNDS*LOCAL_EPOCHS, lr=LR, mu=0.0, seed=SEED)
        locals_models[cid]=(Wl,mu_l,sd_l)
    for cid,(Wl,mu_l,sd_l) in locals_models.items():
        for nm,(Xe,ye) in evalsets.items():
            res.append({'setting':f'local-only[{cid}]','eval':nm, **metrics(ye, predict(Wl,Xe,mu_l,sd_l))})
    print('local-only selesai')
print('=== SEL 4 (local-only) SELESAI ===')

## 5. FEDERATED-SFM (usulan) - simulasi FedAvg in-memory

Loop round persis seperti `fed.run_aggregator`+`run_client`, tapi bobot
dipertukarkan di memori (bukan S3). Tiap klien: z-score lokal, `fed.train_local`
LOCAL_EPOCHS; server: `fed.fedavg` berbobot ukuran data. `mu=0` (FedAvg murni;
FedProx diuji di nb06). Rekam MCC GLOBAL-test tiap round untuk kurva konvergensi.

In [ ]:
if ok:
    # pra-hitung z-score lokal tiap klien (tetap selama training; privasi)
    clients={'cic':(Xc_tr,yc_tr),'unsw':(Xu_tr,yu_tr)}
    zc={cid: fed.zscore_fit(Xl) for cid,(Xl,yl) in clients.items()}
    Xz_cli={cid: fed.zscore_apply(clients[cid][0], *zc[cid]) for cid in clients}
    sizes={cid: len(clients[cid][1]) for cid in clients}
    Wg=fed.init_weights(SEED)
    Xg_te=np.vstack([Xc_te,Xu_te]); yg_te=np.concatenate([yc_te,yu_te])
    curve=[]
    for t in range(ROUNDS):
        cw=[]; cs=[]
        for cid in clients:
            Wl=fed.train_local(Wg, Xz_cli[cid], clients[cid][1],
                               epochs=LOCAL_EPOCHS, lr=LR, mu=0.0, W_global=Wg, seed=SEED)
            cw.append(Wl); cs.append(sizes[cid])
        Wg=fed.fedavg(cw, cs)
        # evaluasi GLOBAL-test memakai z-score GABUNGAN hanya untuk monitoring kurva
        mu_g,sd_g=fed.zscore_fit(np.vstack([Xc_tr,Xu_tr]))
        yp=predict(Wg, Xg_te, mu_g, sd_g)
        curve.append({'round':t, 'MCC_global':round(matthews_corrcoef(yg_te,yp),4)})
    print('federated selesai | rounds=',ROUNDS)
print('=== SEL 5 (federated) SELESAI ===')

## 6. Evaluasi akhir federated + simpan hasil & kurva

**Catatan evaluasi (jujur):** untuk model federated, prediksi tiap test set
memakai z-score **klien sumber** test itu (CIC-test -> z-score CIC; UNSW-test ->
z-score UNSW); GLOBAL-test memakai z-score gabungan train (monitoring). Ini
mencerminkan deployment: tiap node menormalisasi inputnya dengan statistik lokalnya.

In [ ]:
if ok:
    mu_cic,sd_cic=zc['cic']; mu_un,sd_un=zc['unsw']
    fed_eval={'CIC-test':(Xc_te,yc_te,mu_cic,sd_cic),
              'UNSW-test':(Xu_te,yu_te,mu_un,sd_un),
              'GLOBAL-test':(Xg_te,yg_te,*fed.zscore_fit(np.vstack([Xc_tr,Xu_tr])))}
    for nm,(Xe,ye,mu_e,sd_e) in fed_eval.items():
        res.append({'setting':'federated-SFM','eval':nm, **metrics(ye, predict(Wg,Xe,mu_e,sd_e))})
    dfres=pd.DataFrame(res)
    dfres.to_csv(os.path.join(OUTDIR,'fedavg_sfm.csv'),index=False)
    pd.DataFrame(curve).to_csv(os.path.join(OUTDIR,'fedavg_sfm_curve.csv'),index=False)
    import IPython.display as ipd; ipd.display(dfres)
    print('\nKurva konvergensi (MCC global vs round) -> fedavg_sfm_curve.csv')
    print('CEK ekspektasi urutan (H2): centralized >= federated-SFM >= local-only pada GLOBAL-test.')
else:
    print('(lewati: partisi belum ada - jalankan nb01)')
print('=== SEL 6 (hasil federated) SELESAI ===')

## 7. Catatan untuk naskah

- Hasil -> **Tabel D** (`documentation_federated.md`): centralized / local-only /
  federated-SFM pada CIC/UNSW/GLOBAL test. Isi angka NYATA setelah run.
- **H2** diuji di sini (urutan centralized >= federated >= local-only). Besar
  selisih = seberapa jauh FL menutup gap tanpa memusatkan data.
- Kurva `fedavg_sfm_curve.csv` -> Gambar konvergensi (`figure-fed/`, teks Inggris).
- **Bukti SFM = enabler (H1)** ada di nb04 (ablasi dgn vs tanpa SFM), bukan di sini.
- Logika FL identik dgn `aws/fed_node.py` -> hasil simulasi ini memvalidasi
  skrip EC2 sebelum deploy (hemat biaya).